# Lunar Lander PPO Agent Training
This notebook trains a PPO agent on LunarLander-v3 using Stable-Baselines3 and then evaluates its performance.

In [1]:
import os
import numpy as np
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

## 1. Environment Setup

In [2]:
ENV_ID = "LunarLander-v3"
LOG_DIR = "logs"
MODEL_PATH = "lunarlander_ppo"

os.makedirs(LOG_DIR, exist_ok=True)

# 8 vectorized environments for fast rollout collection
n_envs = 8
train_env = make_vec_env(ENV_ID, n_envs=n_envs)
eval_env = Monitor(gym.make(ENV_ID))

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=LOG_DIR,
    log_path=LOG_DIR,
    eval_freq=max(10_000 // n_envs, 1),
    n_eval_episodes=20,
    deterministic=True,
    render=False,
)

## 2. Model Training

In [ ]:
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    n_steps=1024,
    batch_size=64,
    n_epochs=4,
    gamma=0.999,
    gae_lambda=0.98,
    ent_coef=0.01,
    learning_rate=3e-4,
    clip_range=0.2,
    verbose=1,
    tensorboard_log=LOG_DIR,
)

# Reduce total_timesteps if you want a faster test
model.learn(total_timesteps=500_000, callback=eval_callback)
model.save(MODEL_PATH)
print(f"Saved final model to {MODEL_PATH}.zip")

Using cpu device
Logging to logs\PPO_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 88.4     |
|    ep_rew_mean     | -167     |
| time/              |          |
|    fps             | 3829     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 8192     |
---------------------------------
Eval num_timesteps=10000, episode_reward=-963.62 +/- 596.06
Episode length: 153.20 +/- 56.36
----------------------------------------
| eval/                   |            |
|    mean_ep_length       | 153        |
|    mean_reward          | -964       |
| time/                   |            |
|    total_timesteps      | 10000      |
| train/                  |            |
|    approx_kl            | 0.00733234 |
|    clip_fraction        | 0.0118     |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.38      |
|    explained_variance   | -0.000388  |
|    learning_rate        | 0.0003  

## 3. Evaluation and Rendering

In [ ]:
# Load the trained model
model = PPO.load(MODEL_PATH)

episodes = 5
# Use render_mode="human" to watch the lander in a popup window
env = gym.make(ENV_ID, render_mode="human")

rewards = []
for ep in range(episodes):
    state, _ = env.reset()
    total_reward = 0.0
    done = False
    while not done:
        action, _ = model.predict(state, deterministic=True)
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        total_reward += reward
    rewards.append(total_reward)

env.close()
print(f"Mean reward over {episodes} episodes: {np.mean(rewards):.2f} +/- {np.std(rewards):.2f}")

Mean reward over 5 episodes: 190.93 +/- 53.12
